In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
import numpy as np
import pandas as pd
from anndata import AnnData
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import scanpy as sc
from spatialdata_io import xenium
import spatialdata as sd
import os
from scipy.sparse import csr_matrix
print(dega.__version__)

objc[83996]: Class GNotificationCenterDelegate is implemented in both /opt/homebrew/Cellar/glib/2.84.3/lib/libgio-2.0.0.dylib (0x31f7244b8) and /Users/jishar/anaconda3/lib/libgio-2.0.0.dylib (0x31f2da310). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


0.13.0a9


In [3]:
from ipywidgets import Widget
Widget.close_all()

## Real Data

In [4]:
data_dir = "data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs"

## Make AnnData

In [5]:
# # Ingest xenium data raw output folder using spatialdata-io
# sdata = xenium(data_dir)

# # Write sdata to a zarr file
# zarr_path = f"{data_dir}.zarr"

# # Check if the zarr file already exists
# if os.path.exists(zarr_path):
#     print(f"The file {zarr_path} already exists.")
# else:
#     # If the file does not exist, write the data
#     sdata.write(zarr_path)
#     print(f"Data written to {zarr_path} successfully.")

# # Read zarr file using spatialdata
# sdata = sd.read_zarr(zarr_path)

# # Create anndata from sdata.tables['table]
# adata = sdata.tables["table"]
# adata.write_h5ad(f'{data_dir}.h5ad')

In [6]:
# # Load h5ad file
# adata = sc.read_h5ad(f'{data_dir}.h5ad')
# adata.obs.set_index('cell_id', inplace=True)
# adata

## Scanpy processing

In [7]:
# sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
# sc.pp.filter_cells(adata, min_counts=10)
# sc.pp.filter_genes(adata, min_cells=5)

# adata.X = csr_matrix(adata.X)

# adata.layers["counts"] = adata.X

# sc.pp.normalize_total(adata, inplace=True)

# sc.pp.log1p(adata)

# sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# adata = adata[:, adata.var.highly_variable].copy()

# sc.pp.pca(adata)

# sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=10)

# sc.tl.leiden(adata, resolution=1.0)

In [8]:
# adata.write_h5ad(f'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [9]:
adata = sc.read_h5ad(f'data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [10]:
# Suppose this is your top dataframe from parquet
cluster_df = pd.read_parquet("data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test_2/cell_clusters/cluster.parquet")

# Make sure its index is 'cell_id' (looks like it already is, but double-check)
cluster_df.index.name = "cell_id"

# Then overwrite the 'leiden' column in adata.obs by aligning on index
adata.obs['leiden'] = adata.obs.index.map(cluster_df['cluster'])

In [11]:
# Convert the results to a pandas DataFrame
def get_ranked_genes_df(adata, n_genes=100):
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    dfs = []
    for group in groups:
        df = pd.DataFrame({
            'gene': result['names'][group][:n_genes],
            'logfoldchanges': result['logfoldchanges'][group][:n_genes],
            'pvals': result['pvals'][group][:n_genes],
            'pvals_adj': result['pvals_adj'][group][:n_genes],
            'scores': result['scores'][group][:n_genes],
            'cluster': group
        })
        dfs.append(df)
    return pd.concat(dfs)

## Rank and save marker genes

In [12]:
# # Run ranking (faster)
# sc.tl.rank_genes_groups(adata, groupby="leiden", method="t-test", use_raw=False, show_progress=True)

# # Save markers
# marker_df = get_ranked_genes_df(adata, n_genes=100)
# marker_df.to_csv("data/xenium_data/marker_genes_by_cluster_def_clustering.csv", index=False)

#### 1. Uploaded "marker_genes_by_cluster.csv" on ChatGPT, and asked for tentative cell types.
#### 2. "Predicted_Cell_Types_by_Cluster.csv" has the predicted cell types for each cluster based on the top 10 marker genes, with the cluster number included in the label.

In [13]:
# sc assigned clusters and chatgpt given cell types

# pred_cell_types_df = pd.read_csv("data/xenium_data/Predicted_Cell_Types_by_Cluster_def_clustering.csv")
# pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
# pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
# pred_cell_types_df[:5]

In [14]:
pred_cell_types_df = pd.read_csv("data/xenium_data/Predicted_Cell_Types_by_Cluster_def_clustering.csv")
pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
pred_cell_types_df[:5]

,cluster,predicted_cell_type,category
0,1,Epithelial_1,Epithelial
1,2,Unknown_2,Unknown
2,3,Smooth muscle_3,Smooth muscle
3,4,Epithelial_4,Epithelial
4,5,Macrophage_5,Macrophage


### Make hextiles

In [15]:
data = dega.nbhd._get_gdf_cell(adata)
gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=150)

In [16]:
adata_nbp, gdf_nbhd = dega.nbhd.calc_nbp(data, gdf_nbhd, category="cluster")

Calculating NBP


## Cell-cluster by Hextile using NBHD module methods

In [17]:
# Clustering
sc.pp.normalize_total(adata_nbp, inplace=True)
sc.pp.log1p(adata_nbp)
sc.pp.neighbors(adata_nbp, n_neighbors=10)
sc.tl.leiden(adata_nbp, resolution=1)

In [18]:
population_distribution = pd.DataFrame(
    adata_nbp.X, index=adata_nbp.obs_names, columns=adata_nbp.var_names
)

In [19]:
# Add clustering and proportions to hex GeoDataFrame
gdf_nbhd = gdf_nbhd.set_index("name")
gdf_nbhd["leiden"] = adata_nbp.obs["leiden"].values
gdf_nbhd["niche"] = [f"n-{cluster}" for cluster in adata_nbp.obs["leiden"].values]
gdf_nbhd = gdf_nbhd.join(population_distribution)
gdf_nbhd.reset_index(inplace=True)
gdf_nbhd.head()

,name,geometry,leiden,niche,1,10,11,12,13,14,...,23,24,25,3,4,5,6,7,8,9
0,hex_25,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",0,n-0,0.338975,0.034486,0.000000,0.017392,0.017392,0.000000,...,0.0,0.0,0.017392,0.0,0.084083,0.051293,0.0,0.017392,0.034486,0.0
1,hex_26,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",0,n-0,0.451985,0.000000,0.012903,0.000000,0.000000,0.000000,...,0.0,0.0,0.012903,0.0,0.199489,0.025642,0.0,0.000000,0.000000,0.0
2,hex_27,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",0,n-0,0.376478,0.028171,0.000000,0.000000,0.014185,0.014185,...,0.0,0.0,0.014185,0.0,0.170345,0.028171,0.0,0.041964,0.028171,0.0
3,hex_28,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",0,n-0,0.362905,0.010363,0.000000,0.000000,0.000000,0.010363,...,0.0,0.0,0.000000,0.0,0.255933,0.020619,0.0,0.020619,0.030772,0.0
4,hex_29,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",0,n-0,0.112478,0.133531,0.023530,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.154151,0.133531,0.0,0.000000,0.023530,0.0


In [20]:
# Dissolve to form niches
gdf_niche = dega.nbhd._dissolve_by_category(gdf_nbhd, "leiden")
gdf_niche["name"] = [f"n-{c}" for c in gdf_niche["leiden"]]
gdf_niche.head()

,leiden,geometry,name,niche,1,10,11,12,13,14,...,23,24,25,3,4,5,6,7,8,9
0,0,"MULTIPOLYGON (((2416.19133 846.39656, 2351.239...",n-0,n-0,0.338975,0.034486,0.000000,0.017392,0.017392,0.000000,...,0.0,0.000000,0.017392,0.000000,0.084083,0.051293,0.000000,0.017392,0.034486,0.000000
1,1,"MULTIPOLYGON (((1766.67228 2646.39656, 1701.72...",n-1,n-1,0.135036,0.166127,0.011976,0.000000,0.035507,0.069796,...,0.0,0.000000,0.000000,0.000000,0.113759,0.080969,0.000000,0.176279,0.023811,0.000000
2,2,"MULTIPOLYGON (((987.24941 4896.39656, 922.2975...",n-2,n-2,0.000000,0.000000,0.032790,0.000000,0.125163,0.000000,...,0.0,0.000000,0.000000,0.249461,0.000000,0.048790,0.095310,0.000000,0.000000,0.275103
3,3,"MULTIPOLYGON (((3260.56610 5233.89656, 3195.61...",n-3,n-3,0.000000,0.000000,0.093526,0.000000,0.093526,0.000000,...,0.0,0.128617,0.000000,0.179048,0.000000,0.057158,0.093526,0.000000,0.000000,0.128617
4,4,"MULTIPOLYGON (((77.92274 4896.39656, 12.97083 ...",n-4,n-4,0.000000,0.000000,0.207639,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.325422,0.000000,0.000000,0.000000,0.268264


## SKIP - Clustergram: hextile_nbhd-by-cell_population

In [21]:
# gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
# gdf_nbhd_.set_index('name', inplace=True)
# gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
# gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
# gdf_nbhd_.head()

In [22]:
# meta_col = pd.DataFrame(index=gdf_nbhd_.columns.tolist())
# top_cols = [int(col) for col in gdf_nbhd_.sum(axis=0).sort_values(ascending=False).index]
# predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
# meta_col['category'] = predicted_types
# meta_col[:5]

In [23]:
# meta_row = pd.DataFrame(index=gdf_nbhd_.index.tolist())
# top_rows = gdf_nbhd_.sum(axis=1).sort_values(ascending=False).index.tolist()
# niches = gdf_nbhd.set_index("name").loc[top_rows, "niche"].tolist()
# meta_row['niche'] = niches
# meta_row[:5]

In [24]:
# mat = dega.clust.Matrix(
#     gdf_nbhd_,
#     name='parquet',
#     meta_col=meta_col,
#     col_attr=['category'],
#     meta_row=meta_row
# )

# mat.norm(axis='row', by='zscore')
# mat.clust()
# cgm = dega.viz.Clustergram(
#     matrix=mat, 
#     width=500, 
#     height=500
# )
# cgm

## Clustergram: cell_population-by-hextile_nbhd 

In [25]:
gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
gdf_nbhd_.set_index('name', inplace=True)
gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
gdf_nbhd_.head()

,1,10,11,12,13,14,15,16,17,18,...,23,24,25,3,4,5,6,7,8,9
name,,,,,,,,,,,,,,,,,,,,,
hex_25,0.338975,0.034486,0.000000,0.017392,0.017392,0.000000,0.034486,0.000000,0.0,0.000000,...,0.0,0.0,0.017392,0.0,0.084083,0.051293,0.0,0.017392,0.034486,0.0
hex_26,0.451985,0.000000,0.012903,0.000000,0.000000,0.000000,0.012903,0.000000,0.0,0.025642,...,0.0,0.0,0.012903,0.0,0.199489,0.025642,0.0,0.000000,0.000000,0.0
hex_27,0.376478,0.028171,0.000000,0.000000,0.014185,0.014185,0.028171,0.000000,0.0,0.000000,...,0.0,0.0,0.014185,0.0,0.170345,0.028171,0.0,0.041964,0.028171,0.0
hex_28,0.362905,0.010363,0.000000,0.000000,0.000000,0.010363,0.040822,0.010363,0.0,0.020619,...,0.0,0.0,0.000000,0.0,0.255933,0.020619,0.0,0.020619,0.030772,0.0
hex_29,0.112478,0.133531,0.023530,0.000000,0.000000,0.000000,0.023530,0.046520,0.0,0.046520,...,0.0,0.0,0.000000,0.0,0.154151,0.133531,0.0,0.000000,0.023530,0.0


In [26]:
gdf_nbhd_T = gdf_nbhd_.T
gdf_nbhd_T.head()

name,hex_25,hex_26,hex_27,hex_28,hex_29,hex_30,hex_31,hex_32,hex_33,hex_34,...,hex_5772,hex_5801,hex_5803,hex_5804,hex_5805,hex_5806,hex_5807,hex_5808,hex_5809,hex_5810
1,0.338975,0.451985,0.376478,0.362905,0.112478,0.326397,0.135036,0.240385,0.317535,0.129534,...,0.000000,0.0,0.042560,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.068993
10,0.034486,0.000000,0.028171,0.010363,0.133531,0.017392,0.166127,0.160343,0.068319,0.061875,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.037740,0.000000
11,0.000000,0.012903,0.000000,0.000000,0.023530,0.017392,0.011976,0.021506,0.029853,0.041673,...,0.000000,0.0,0.083382,0.117783,0.036368,0.033902,0.154151,0.00000,0.109199,0.000000
12,0.017392,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
13,0.017392,0.000000,0.014185,0.000000,0.000000,0.000000,0.035507,0.000000,0.000000,0.000000,...,0.223144,0.0,0.000000,0.060625,0.169899,0.346276,0.000000,0.09531,0.074108,0.000000


In [27]:
meta_col = pd.DataFrame(index=gdf_nbhd_T.columns.tolist())
top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index.tolist()
niches = gdf_nbhd.set_index("name").loc[top_cols, "niche"].tolist()
meta_col['niche'] = niches
meta_col[:5]

,niche
hex_25,n-17
hex_26,n-17
hex_27,n-17
hex_28,n-17
hex_29,n-17


In [28]:
meta_row = pd.DataFrame(index=gdf_nbhd_T.index.tolist())
top_rows = [int(row) for row in gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index]
predicted_types = pred_cell_types_df.set_index("cluster").loc[top_rows, "category"].tolist()
meta_row['category'] = predicted_types
meta_row[:5]

,category
1,Unknown
10,Smooth muscle
11,Macrophage
12,Endothelial
13,Smooth muscle


In [29]:
# # Transpose neighborhood matrix
# gdf_nbhd_T = gdf_nbhd_.T

# # Rename index and column names
# gdf_nbhd_T.index.name = "cluster"
# gdf_nbhd_T.columns.name = ""

# # # Ensure cluster index is integer
# # gdf_nbhd_T.index = gdf_nbhd_T.index.astype(int)

# # # Create cluster → cell type mapping
# # cluster_to_cell_type = pred_cell_types_df.set_index("cluster")["predicted_cell_type"].to_dict()

# # # Map cluster index to predicted cell type
# # gdf_nbhd_T.index = gdf_nbhd_T.index.map(cluster_to_cell_type)

# # # Check for unmapped values
# # if gdf_nbhd_T.index.isnull().any():
# #     raise ValueError("Some clusters could not be mapped to predicted cell types.")

# # Build metadata for rows
# top_rows = gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index
# predicted_types_df = pred_cell_types_df.set_index("predicted_cell_type")

# # Filter and align with top rows (cell types)
# predicted_types = predicted_types_df.loc[top_rows, "category"]
# meta_row = pd.DataFrame(index=top_rows)
# meta_row['category'] = predicted_types.values

# # Build metadata for columns
# top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index
# niches_df = gdf_nbhd.set_index("name")

# # Filter and align with top columns (neighborhoods)
# niches = niches_df.loc[top_cols, "niche"]
# meta_col = pd.DataFrame(index=top_cols)
# meta_col['niche'] = niches.values

In [30]:
mat = dega.clust.Matrix(
    gdf_nbhd_T,
    name='parquet',
    meta_col=meta_col,
    col_attr=['niche'],
    meta_row=meta_row
)

mat.downsample_to(axis='col', category='niche')
mat.norm(axis='row', by='zscore')
mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500
)
# cgm

## Visualize in Landscape view: Hextile NBHD

In [31]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF = gdf_nbhd_LF[['geometry','name','leiden']]
gdf_nbhd_LF.rename(columns={'leiden':'cat'}, inplace=True)
gdf_nbhd_LF.head()

,geometry,name,cat
0,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",hex_25,0
1,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",hex_26,0
2,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",hex_27,0
3,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",hex_28,0
4,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",hex_29,0


In [32]:
categories = gdf_nbhd_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_nbhd_LF['color'] = gdf_nbhd_LF['cat'].astype(str).map(cat_to_hex)
gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area
gdf_nbhd_LF["cat"] = [f"n-{c}" for c in gdf_nbhd_LF["cat"]]
gdf_nbhd_LF.head()

/var/folders/_6/bhs42vt57t1dkb59k4sy0p440000gp/T/ipykernel_83996/1160049529.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area


,geometry,name,cat,color,area
0,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",hex_25,n-0,#1f77b4,14614.178689
1,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",hex_26,n-0,#1f77b4,14614.178689
2,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",hex_27,n-0,#1f77b4,14614.178689
3,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",hex_28,n-0,#1f77b4,14614.178689
4,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",hex_29,n-0,#1f77b4,14614.178689


In [33]:
sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
data_dir = f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/{sample}_test'
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

# base_url = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_v2/main/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url = base_url,
    nbhd=gdf_nbhd_LF
)

# landscape_ist

In [ ]:
dega.viz.landscape_clustergram(landscape_ist, cgm, entity="NBHD")

In [35]:
def _get_name_mapping(path_landscape_files, layer, segmentation="default"):
    """
    Generates mappings from gene and cell names to unique integer identifiers.

    Args:
        path_landscape_files (str): Path to the directory containing the metadata files.
            Expected files:
            - `meta_gene.parquet`: Contains gene metadata with gene names as the index.
            - `cell_metadata.parquet`: Contains cell metadata with a 'name' column.
            - `layer`: 'boundary' or 'transcript'
            - `segmentation`: 'default' or 'cellpose2', etc.

    Returns:
        dict: Maps gene names (str) to integer ranks (int).
    """

    if layer == "transcript":
        # Load gene metadata
        df_meta_gene = pd.read_parquet(f"{path_landscape_files}/meta_gene.parquet")
        if segmentation != "default":
            df_meta_gene = pd.read_parquet(
                f"{path_landscape_files}/meta_gene_{segmentation}.parquet"
            )
        df_meta_gene["name"] = df_meta_gene.index
        df_meta_gene = df_meta_gene.reset_index(drop=True)
        return {str(name): idx for idx, name in df_meta_gene["name"].items()}

    if layer == "boundary":
        # Load cell metadata
        df_meta_cell = pd.read_parquet(f"{path_landscape_files}/cell_metadata.parquet")
        if segmentation != "default":
            df_meta_cell = pd.read_parquet(
                f"{path_landscape_files}/cell_metadata_{segmentation}.parquet"
            )
        return {str(name): idx for idx, name in df_meta_cell["name"].items()}

In [ ]:
# convert cell index from string to integer
cell_str_to_int_mapping = dega._get_name_mapping(
    base_path, layer="boundary", segmentation=segmentation_approach
)



In [ ]:
gdf_nbhd_TEST = gdf_nbhd_.copy()
print(gdf_nbhd_TEST.head())
gdf_nbhd_TEST["name"] = gdf_nbhd_TEST.index
gdf_nbhd_TEST = gdf_nbhd_TEST.reset_index(drop=True)
dict_mapped = {str(name): idx for idx, name in gdf_nbhd_TEST["name"].items()}

In [ ]:
from pathlib import Path
output_dir = "data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test/cbg"
output_dir = Path(output_dir)

In [ ]:
dict_mapped['name']

In [ ]:
gdf_nbhd_TEST

In [ ]:
# gdf_nbhd_TEST.index = gdf_nbhd_TEST.index.map(dict_mapped)

for index, nbhd in enumerate(gdf_nbhd_TEST.columns):

    print(f"Processing gene {index}: {nbhd}")

    # Extract the column as a DataFrame as a copy
    col_df = gdf_nbhd_TEST[[nbhd]].copy()

    # Create a DataFrame necessary to prevent error in to_parquet
    inst_df = pd.DataFrame(col_df.values, columns=[nbhd], index=col_df.index.tolist())

    # Replace 0 with NA and drop rows where all values are NA
    inst_df.replace(0, pd.NA, inplace=True)
    inst_df.dropna(how="all", inplace=True)

    # Save to Parquet if DataFrame is not empty
    if not inst_df.empty:
        output_path = output_dir / f"{nbhd}.parquet"
        inst_df.to_parquet(output_path)

In [ ]:
pd.read_parquet("data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test/cell_clusters/cluster.parquet").head()

In [ ]:
pd.read_parquet("data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test/meta_gene.parquet").head()

In [ ]:
pd.read_parquet("data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test/cbg/2.parquet")

In [ ]:
pd.read_parquet("data/landscape_files/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_test/cbg/AAMP.parquet")

## SKIP - Visualize in Landscape view: Niche NBHD

In [ ]:
gdf_niche_LF = gdf_niche.copy()
gdf_niche_LF = gdf_niche_LF[['geometry','name','leiden']]
gdf_niche_LF.rename(columns={'leiden':'cat'}, inplace=True)

# Convert 'cat' from categorical strings like '0' → int → +1 → str again
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(int) + 1
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(str)
# gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype('category')

gdf_niche_LF.head()

In [ ]:
categories = gdf_niche_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_niche_LF['color'] = gdf_niche_LF['cat'].astype(str).map(cat_to_hex)
gdf_niche_LF.head()

In [ ]:
sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
data_dir = f'data/xenium_data/'
path_landscape_files=f'data/landscape_files/{sample}_test'

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
    nbhd=gdf_niche_LF
)

# landscape_ist

## SKIP - Clustergram: niche_nbhd-by-cell_population

In [ ]:
gdf_niche_ = gdf_niche.drop(['geometry', 'leiden'], axis=1)
gdf_niche_.set_index('name', inplace=True)

gdf_niche_ = gdf_niche_.apply(pd.to_numeric, errors='coerce')
gdf_niche_ = gdf_niche_.replace([np.inf, -np.inf], np.nan).fillna(0)
gdf_niche_ = gdf_niche_[(gdf_niche_ != 0).any(axis=1)]
gdf_niche_ = gdf_niche_.loc[gdf_niche_.std(axis=1) != 0]
gdf_niche_ = gdf_niche_.loc[:, gdf_niche_.std(axis=0) != 0]

assert np.isfinite(gdf_niche_.values).all(), "Matrix still contains non-finite values!"

In [ ]:
meta_col = pd.DataFrame(index=gdf_niche_.columns[1:].tolist())
top_cols = [int(col) for col in gdf_niche_.sum(axis=0).sort_values(ascending=False).index[1:]]
predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
meta_col['category'] = predicted_types
meta_col[:5]

In [ ]:
gdf_niche_.set_index('niche', inplace=True)

In [ ]:
gdf_niche_

In [ ]:
mat = dega.clust.Matrix(
    gdf_niche_,
    name='parquet',
    meta_col=meta_col,
    col_attr=['category'],
)

mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500
)
cgm